# 00 — Phase-0 pipeline map

The Phase-0 flowchart as a live diagram: each node is one pipeline (and one
notebook in this series), colored by its status parsed **live** from
`docs/phase0-checklist.md`. This is the index for notebooks 01–11.

Exploratory only — the statuses and numbers are those recorded in the
checklist / RESULTS.md; this notebook renders them, it does not measure.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx

import nbsupport as nbs

nbs.style()

In [ ]:
CL = nbs.parse_checklist()
nbs.provenance_header(
    "00",
    "Phase-0 pipeline map",
    "done" if all(CL["rows"].get(r) for r in (1, 6, 7, 8, 9, 10, 11)) else "partial",
    results_rows=[
        "docs/phase0-checklist.md — the living measurement program (rows 1–15)",
        "docs/phase0-killtest-verdict.md — the rendered kill-test verdict (2026-07-12)",
    ],
    data=["docs/phase0-checklist.md (parsed live)"],
    scripts=["(this notebook renders the checklist; producers are per-node, see 01–11)"],
)

## The pipeline DAG

Nodes are laid out left→right in dependency order. Status comes from the
checklist rows each node is responsible for (`NODE_ROWS`); nodes with no
checklist row of their own (data acquisition, NNN repro, rate cross-check,
rerun campaign) are keyed to the "local confirmations" bullets or marked
done from their RESULTS.md rows. `dot` is not installed, so positions are
hardcoded (robust, no layout backend needed).

In [ ]:
# node_id: (label, x-layer, y-lane, checklist rows, local-confirmation substrings)
NODES = {
    "01": ("01 data\ninventory", 0, 2.6, [], ["Exact mesa_80/mesa_151 isotope lists"]),
    "02": ("02 NNN\nbaseline", 1, 3.6, [], []),  # RESULTS-only, exact repro
    "03": ("03 graph /\nconservation", 1, 2.6, [1, 7, 10], []),
    "04": ("04 rates\ncrosscheck", 1, 1.6, [], [
        "Softwired inverse-rate equilibrium", "Which weak-rate tables",
        "pynucastro TabularRate precedence"]),
    "07": ("07 trajectory\ningestion", 1, 0.4, [11], []),
    "05": ("05 flux\nengine", 2, 2.6, [6], []),
    "06": ("06 QSE /\nbridges", 3, 3.2, [6], []),
    "08": ("08 rerun\ncampaign", 2, 0.9, [], []),  # RESULTS-only, 418/418 OK
    "09": ("09 label\npathology", 3, 0.2, [8], []),
    "10": ("10 kill-test\nverdict", 4, 1.9, [6, 7, 8, 9], []),
    "11": ("11 integrator\n+ handoff", 5, 1.9, [11, 2, 3, 4, 5, 12, 13, 14, 15], []),
}
EDGES = [
    ("01", "02"), ("01", "03"), ("01", "04"), ("01", "07"),
    ("03", "05"), ("04", "05"), ("05", "06"), ("05", "10"),
    ("07", "08"), ("07", "09"), ("06", "10"), ("08", "10"),
    ("09", "10"), ("10", "11"),
]

# Nodes 02 and 08 are RESULTS-only (no checklist row of their own); mark done explicitly.
FORCED = {"02": "done", "08": "done"}

#: KNOWN CONFLICT (surfaced, not silently resolved — docs/CLAUDE.md Rule 0):
#: node 01 keys to the local-confirmation bullet "Exact mesa_80/mesa_151 isotope
#: lists", still [ ] in docs/phase0-checklist.md, while RESULTS.md 2026-07-08
#: carries both the isotope counts (80/151) and the full Yₑ-controller membership
#: table. The map therefore renders 01 as NOT BUILT. Either the bullet is stale
#: (Rule 3: tick it, naming the rows) or it wants something the rows do not yet
#: cover — a human decides; this notebook only reports the disagreement.
CONFLICTS = {
    "01": "checklist bullet unticked, but RESULTS.md 2026-07-08 has the measurement",
}

STATUS_COLOR = {
    "done": "#009E73",
    "partial": "#E69F00",
    "unbuilt": "#BBBBBB",
    "unknown": "#FFFFFF",
}


def node_status(nid):
    if nid in FORCED:
        return FORCED[nid]
    _, _, _, rows, local = NODES[nid]
    return nbs.status_of(rows, local, checklist=CL)


g = nx.DiGraph()
pos = {}
colors = []
labels = {}
for nid, (label, x, y, _, _) in NODES.items():
    g.add_node(nid)
    pos[nid] = (x, y)
    labels[nid] = label
    colors.append(STATUS_COLOR[node_status(nid)])
g.add_edges_from(EDGES)

fig, ax = plt.subplots(figsize=(13, 6.5))
nx.draw_networkx_edges(
    g, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14,
    edge_color="#888888", node_size=6500, min_target_margin=22,
)
nx.draw_networkx_nodes(
    g, pos, ax=ax, node_color=colors, node_size=6500,
    edgecolors="#333333", linewidths=1.3, node_shape="s",
)
nx.draw_networkx_labels(g, pos, labels, ax=ax, font_size=8.5)
for nid in CONFLICTS:
    x_, y_ = pos[nid]
    ax.annotate("⚠ checklist/RESULTS\nconflict — see below", (x_, y_ - 0.42),
                ha="center", va="top", fontsize=6.5, color="#B3261E")
ax.set_title("Phase-0 pipeline flowchart — node = pipeline (and notebook), color = checklist status")
ax.set_xlim(-0.6, 5.6)
ax.set_ylim(-0.4, 4.2)
ax.axis("off")
legend = [
    mpatches.Patch(color=c, label=s.upper()) for s, c in STATUS_COLOR.items()
    if s != "unknown"
]
ax.legend(handles=legend, loc="upper left", ncol=3, fontsize=9)
nbs.caption(
    fig,
    "All three Phase-0 charter deliverables (NNN repro, graph+conservation, kill-test) are "
    "done; the integrator node is partial (hot strata censored) and the Step-7/Phase-1 nodes "
    "downstream of it are not built. Status is parsed live from the checklist — if a row's "
    "checkbox flips, so does its node here. Node 01 renders NOT BUILT on a checklist bullet "
    "that RESULTS.md 2026-07-08 appears to have already measured: an unresolved "
    "checklist-vs-RESULTS conflict, reported rather than papered over (docs/CLAUDE.md Rule 0).",
    results=["docs/phase0-checklist.md rows 1–15", "docs/phase0-killtest-verdict.md"],
    scripts=["(per-node producers listed in notebooks 01–11)"],
)

## Checklist rows 1–15 at a glance

In [ ]:
rows = sorted(CL["rows"])
done = [CL["rows"][r] for r in rows]
fig, ax = plt.subplots(figsize=(11, 2.6))
ax.bar(rows, [1] * len(rows), color=["#009E73" if d else "#BBBBBB" for d in done])
for r, d in zip(rows, done):
    ax.text(r, 0.5, "✓" if d else "·", ha="center", va="center",
            fontsize=13, color="white" if d else "#555555")
ax.set_xticks(rows)
ax.set_yticks([])
ax.set_xlabel("phase0-checklist.md row")
ax.set_title(f"Measurement rows: {sum(done)}/{len(rows)} measured")
ax.grid(False)
nbs.caption(
    fig,
    "Rows 1,6–11 measured (graph, κ, cond, Guidry sweep, churn, projector, energy); rows "
    "2–5 and 12–15 are model-dependent Phase-1 work (features, ablations, transfer, the "
    "Yₑ accumulation-slope pass/fail gate). Row 12 is the single biggest lever.",
    results=["docs/phase0-checklist.md rows 1–15"],
    scripts=["(this notebook parses the checklist)"],
)

## Notebook index

| # | Node | Notebook |
|---|------|----------|
| 00 | pipeline map | `00-phase0-map.ipynb` (this) |
| 01 | data inventory | `01-data-inventory.ipynb` |
| 02 | NNN baseline | `02-nnn-baseline.ipynb` |
| 03 | graph / conservation | `03-graph-conservation.ipynb` |
| 04 | rates cross-check | `04-rates-crosscheck.ipynb` |
| 05 | flux engine | `05-flux-engine.ipynb` |
| 06 | QSE / bridges | `06-qse-bridges.ipynb` |
| 07 | trajectory ingestion | `07-trajectories.ipynb` |
| 08 | rerun campaign | `08-rerun-campaign.ipynb` |
| 09 | label pathology | `09-label-pathology.ipynb` |
| 10 | kill-test verdict | `10-killtest-verdict.ipynb` |
| 11 | integrator + handoff | `11-integrator-and-handoff.ipynb` |